# 02 — Chunking strategy experiment

Compare page-aware vs heading-aware chunking on a 5-PDF sample. Pick the winner, then chunk the full corpus into `data/processed/{ticker}/{year}.jsonl`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import random
from src.ingestion.pdf_parser import parse_pdf
from src.ingestion.chunker import chunk_pages

random.seed(42)
RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

In [ ]:
# Pick a 5-PDF sample
all_pdfs = sorted(RAW.rglob("*.pdf"))
sample = random.sample(all_pdfs, min(5, len(all_pdfs)))
sample_meta = [(p.parent.name, int(p.stem), p) for p in sample]
sample_meta

In [ ]:
# Strategy A: page-aware (default chunker, max_tokens=512, overlap=50)
chunks_A = []
for ticker, year, p in sample_meta:
    pages = list(parse_pdf(p, ticker=ticker, year=year))
    chunks_A.extend(chunk_pages(pages, max_tokens=512, overlap=50))
print(f"Strategy A (page-aware 512/50): {len(chunks_A)} chunks")

In [ ]:
# Strategy B: heading-aware via unstructured (sketches — adopted only if it wins).
# Falls back gracefully if unstructured can't open a given PDF.
from unstructured.partition.pdf import partition_pdf

def heading_aware_chunks(pdf_path, ticker, year, max_tokens=512):
    chunks = []
    try:
        elements = partition_pdf(filename=str(pdf_path), strategy="fast")
    except Exception as e:
        print(f"  unstructured failed on {pdf_path.name}: {e}")
        return []
    current_heading = None
    buf = []
    page = 1
    for el in elements:
        text = el.text or ""
        if el.category in ("Title", "Header"):
            if buf:
                chunks.append({"text": " ".join(buf), "ticker": ticker, "year": year, "page": page, "section": current_heading or ""})
                buf = []
            current_heading = text
        else:
            buf.append(text)
            page = getattr(el.metadata, "page_number", page) or page
            if sum(len(x.split()) for x in buf) >= max_tokens:
                chunks.append({"text": " ".join(buf), "ticker": ticker, "year": year, "page": page, "section": current_heading or ""})
                buf = []
    if buf:
        chunks.append({"text": " ".join(buf), "ticker": ticker, "year": year, "page": page, "section": current_heading or ""})
    return chunks

chunks_B = []
for ticker, year, p in sample_meta:
    chunks_B.extend(heading_aware_chunks(p, ticker, year))
print(f"Strategy B (heading-aware): {len(chunks_B)} chunks")

## Choice

Empirical comparison on a small manual set (qa_pilot.jsonl) belongs in notebook 04. For now we go with **Strategy A (page-aware, 512 tokens, 50 overlap)** as the default — it preserves citations cleanly without depending on `unstructured`'s heading detection (which is unreliable on financial-filing layouts). Switch to B only if notebook 04 demonstrates a meaningful recall@5 win.

In [ ]:
# Materialise the full corpus with Strategy A — one JSONL per PDF
written = 0
for p in sorted(RAW.rglob("*.pdf")):
    ticker = p.parent.name
    year = int(p.stem)
    pages = list(parse_pdf(p, ticker=ticker, year=year))
    chunks = list(chunk_pages(pages, max_tokens=512, overlap=50))
    out_dir = PROCESSED / ticker
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{year}.jsonl"
    with out_path.open("w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps(c) + "\n")
    written += 1

print(f"Wrote chunks for {written} PDFs to {PROCESSED}")